In [7]:
def weighted_sum(x, weight):
    return (weight * x).sum(axis=-1)

### addition

In [8]:
from einops import rearrange
import torch

import triton
import triton.language as tl

In [4]:
@triton.jit
def add_kernel(
    x_ptr,  # *Pointer* to first input vector.
    y_ptr,  # *Pointer* to second input vector.
    output_ptr,  # *Pointer* to output vector.
    n_elements,  # Size of the vector.
    BLOCK_SIZE: tl.constexpr,  # Number of elements each program should process.
                 # NOTE: `constexpr` so it can be used as a shape value.
):
    # There are multiple 'programs' processing different data. We identify which program
    # we are here:
    pid = tl.program_id(axis=0)  # We use a 1D launch grid so axis is 0.
    block_start = pid * BLOCK_SIZE
    offsets = block_start + tl.arange(0, BLOCK_SIZE)
    mask = offsets < n_elements
    # Load x and y from DRAM, masking out any extra elements in case the input is not a
    # multiple of the block size.
    x = tl.load(x_ptr + offsets, mask=mask)
    y = tl.load(y_ptr + offsets, mask=mask)
    output = x + y
    # Write x + y back to DRAM.
    tl.store(output_ptr + offsets, output, mask=mask)

In [5]:
def add(x: torch.Tensor, y: torch.Tensor):
    # We need to preallocate the output.
    output = torch.empty_like(x)
    assert x.is_cuda and y.is_cuda and output.is_cuda
    n_elements = output.numel()
    print(n_elements)
    # The SPMD launch grid denotes the number of kernel instances that run in parallel.
    # It is analogous to CUDA launch grids. It can be either Tuple[int], or Callable(metaparameters) -> Tuple[int].
    # In this case, we use a 1D grid where the size is the number of blocks:
    def grid(meta):
        print(meta)
        print((triton.cdiv(n_elements, meta['BLOCK_SIZE']),))
        return (triton.cdiv(n_elements, meta['BLOCK_SIZE']),)
    
    # NOTE:
    #  - Each torch.tensor object is implicitly converted into a pointer to its first element.
    #  - `triton.jit`'ed functions can be indexed with a launch grid to obtain a callable GPU kernel.
    #  - Don't forget to pass meta-parameters as keywords arguments.
    add_kernel[grid](x, y, output, n_elements, BLOCK_SIZE=1024)
    return output

In [6]:
x = torch.zeros(size = [2048], device = "cuda")
y = torch.zeros(size = [2048], device = "cuda")
add(x,y)

2048
{'x_ptr': tensor([0., 0., 0.,  ..., 0., 0., 0.], device='cuda:0'), 'y_ptr': tensor([0., 0., 0.,  ..., 0., 0., 0.], device='cuda:0'), 'output_ptr': tensor([0., 0., 0.,  ..., 0., 0., 0.], device='cuda:0'), 'n_elements': 2048, 'BLOCK_SIZE': 1024}
(2,)


tensor([0., 0., 0.,  ..., 0., 0., 0.], device='cuda:0')

### Weighted Sum

In [ ]:
from cs336_systems.weighted_sum import WeightedSumFunc

In [23]:
x = torch.zeros(size = [100,120], device = "cuda")
weight = torch.zeros(size = [120], device = "cuda")
WeightedSumFunc.apply(x, weight)

128 8 16


tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0.], device='cuda:0')